In [1]:
# Cell 1 — Load the canonical dataset configuration and metadata.
#
# This notebook starts independently of the previous notebooks.
# It reads the saved build summary rather than relying on
# variables left in another notebook's kernel.
#
# No market-data rows are loaded or changed yet.

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


def find_project_root(start_path):
    """Locate the repository from the notebook's working directory."""

    current_path = Path(start_path).resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / ".git").exists()
            and (candidate / "raw_data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

REPORT_TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"


# This project currently has one canonical build summary.
# When additional assets or dataset versions are added, we will
# select their summary explicitly instead of silently choosing one.
summary_candidates = sorted(
    REPORT_TABLES_DIR.glob("*_build_summary.json")
)

if len(summary_candidates) != 1:
    raise ValueError(
        "Expected one build summary. Available summaries:\n"
        + "\n".join(str(path) for path in summary_candidates)
    )

BUILD_SUMMARY_FILE = summary_candidates[0]

with BUILD_SUMMARY_FILE.open(
    mode="r",
    encoding="utf-8",
) as summary_handle:
    build_summary = json.load(summary_handle)


instrument_config = build_summary["instrument_config"]

CANONICAL_INPUT_FILE = (
    PROJECT_ROOT
    / build_summary["dataset"]["relative_path"]
)

BAR_MINUTES = int(instrument_config["bar_minutes"])

FORECAST_HORIZON_MINUTES = int(
    instrument_config["forecast_horizon_minutes"]
)

if BAR_MINUTES <= 0 or FORECAST_HORIZON_MINUTES <= 0:
    raise ValueError("Bar and forecast durations must be positive.")

if FORECAST_HORIZON_MINUTES % BAR_MINUTES != 0:
    raise ValueError(
        "The forecast horizon must align with the bar duration."
    )

if build_summary["status"] != "PASS_WITH_COVERAGE_WARNINGS":
    raise ValueError("Unexpected canonical build status.")


with pq.ParquetFile(CANONICAL_INPUT_FILE) as canonical_file:
    input_row_count = canonical_file.metadata.num_rows
    input_column_names = canonical_file.schema_arrow.names

assert (
    input_row_count
    == build_summary["dataset"]["observed_bars"]
), "Dataset row count differs from the build summary."


print("Notebook setup: PASS")
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

print()
print("Instrument:", instrument_config["symbol"])
print("Bar duration:", BAR_MINUTES, "minute(s)")
print("Forecast horizon:", FORECAST_HORIZON_MINUTES, "minutes")
print("Canonical bars:", f"{input_row_count:,}")
print(
    "Recorded coverage gaps:",
    f"{build_summary['coverage']['gap_count']:,}",
)

print()
print("Input dataset:", CANONICAL_INPUT_FILE)
print("Build summary:", BUILD_SUMMARY_FILE)

Notebook setup: PASS
Python: 3.13.16
Pandas: 3.0.6
PyArrow: 25.0.1

Instrument: EURUSD
Bar duration: 1 minute(s)
Forecast horizon: 10 minutes
Canonical bars: 6,206,712
Recorded coverage gaps: 15,222

Input dataset: C:\Users\Ijzd4\trend models\EURUSD 15\data\interim\eurusd_1min_2010-01-01_2026-10-06.parquet
Build summary: C:\Users\Ijzd4\trend models\EURUSD 15\reports\tables\eurusd_1min_2010-01-01_2026-10-06_build_summary.json


In [2]:
# Cell 2 — Build the timeline and identify continuous segments.
#
# Load only timestamps, not the full price dataset.
# A completed bar becomes usable at its "available_at" timestamp.
# We call this the decision time.
#
# Every gap starts a new segment.
# Later, lookback windows and future targets must remain
# within the same segment. We do not fill missing minutes.

BAR_DURATION = pd.Timedelta(minutes=BAR_MINUTES)

canonical_timeline = pd.read_parquet(
    CANONICAL_INPUT_FILE,
    columns=["minute_start", "available_at"],
).rename(columns={"available_at": "decision_time"})


# Independently verify the timeline before using it.
assert len(canonical_timeline) == input_row_count
assert not canonical_timeline.isna().any().any()
assert canonical_timeline["minute_start"].is_monotonic_increasing
assert canonical_timeline["minute_start"].is_unique

assert (
    canonical_timeline["decision_time"]
    == canonical_timeline["minute_start"] + BAR_DURATION
).all(), "Unexpected bar availability timing."


# The first row starts a segment.
# Any subsequent interval other than one bar starts another.
bar_intervals = canonical_timeline["minute_start"].diff()

segment_starts = bar_intervals.ne(BAR_DURATION)

canonical_timeline["segment_id"] = (
    segment_starts.cumsum().astype("int32") - 1
)


segment_count = int(segment_starts.sum())
gap_count = segment_count - 1

assert gap_count == build_summary["coverage"]["gap_count"], (
    "Timeline gap count differs from the build summary."
)


# Report memory use before loading additional columns.
timeline_memory_mib = (
    canonical_timeline.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

print(f"Timeline rows: {len(canonical_timeline):,}")
print(f"Coverage gaps: {gap_count:,}")
print(f"Continuous segments: {segment_count:,}")
print(f"Timeline memory: {timeline_memory_mib:.2f} MiB")
print()
print("Continuity segmentation: PASS")

display(canonical_timeline.head())


# Release temporary Series; retain the timeline and segment IDs.
del bar_intervals, segment_starts

Timeline rows: 6,206,712
Coverage gaps: 15,222
Continuous segments: 15,223
Timeline memory: 118.38 MiB

Continuity segmentation: PASS


,minute_start,decision_time,segment_id
0,2010-01-01 00:00:00+00:00,2010-01-01 00:01:00+00:00,0
1,2010-01-01 00:01:00+00:00,2010-01-01 00:02:00+00:00,0
2,2010-01-01 00:02:00+00:00,2010-01-01 00:03:00+00:00,0
3,2010-01-01 00:03:00+00:00,2010-01-01 00:04:00+00:00,0
4,2010-01-01 00:04:00+00:00,2010-01-01 00:05:00+00:00,0


In [3]:
# Cell 3 — Identify valid forecast windows.
#
# Forecast timing starts at decision_time, not minute_start.
# Example: a bar available at 14:31 forecasts through 14:41.
#
# Both endpoints must belong to the same continuous segment.
# This guarantees that all intervening one-minute bars exist.
#
# These are target-eligibility checks, NOT model features.
# Future information must never become a model input.

FORECAST_HORIZON_BARS = (
    FORECAST_HORIZON_MINUTES // BAR_MINUTES
)

FORECAST_DURATION = pd.Timedelta(
    minutes=FORECAST_HORIZON_MINUTES
)


canonical_timeline["target_time"] = (
    canonical_timeline["decision_time"] + FORECAST_DURATION
)


# Locate the future endpoint by row offset.
# The continuity checks below prevent an offset from jumping
# over a weekend, missing minute, or the end of the dataset.
future_decision_time = (
    canonical_timeline["decision_time"]
    .shift(-FORECAST_HORIZON_BARS)
)

future_segment_id = (
    canonical_timeline["segment_id"]
    .shift(-FORECAST_HORIZON_BARS)
)

canonical_timeline["target_window_valid"] = (
    future_decision_time.eq(canonical_timeline["target_time"])
    & future_segment_id.eq(canonical_timeline["segment_id"])
)


# Independently reconcile eligibility using segment lengths.
# Each segment loses its final horizon-length set of rows.
segment_lengths = (
    canonical_timeline.groupby("segment_id", sort=False).size()
)

expected_valid_rows = int(
    (segment_lengths - FORECAST_HORIZON_BARS)
    .clip(lower=0)
    .sum()
)

valid_target_rows = int(
    canonical_timeline["target_window_valid"].sum()
)

assert valid_target_rows == expected_valid_rows, (
    "Forecast-window eligibility failed reconciliation."
)


print("Forecast horizon:", FORECAST_HORIZON_MINUTES, "minutes")
print(f"Valid future windows: {valid_target_rows:,}")
print(
    f"Unavailable future windows: "
    f"{len(canonical_timeline) - valid_target_rows:,}"
)
print()
print("Forecast-window validation: PASS")

display(
    canonical_timeline[
        ["decision_time", "target_time", "target_window_valid"]
    ].head()
)


# Final training eligibility will also require feature warmup.
# Do not discard rows yet.
del future_decision_time, future_segment_id, segment_lengths

Forecast horizon: 10 minutes
Valid future windows: 6,087,107
Unavailable future windows: 119,605

Forecast-window validation: PASS


,decision_time,target_time,target_window_valid
0,2010-01-01 00:01:00+00:00,2010-01-01 00:11:00+00:00,True
1,2010-01-01 00:02:00+00:00,2010-01-01 00:12:00+00:00,True
2,2010-01-01 00:03:00+00:00,2010-01-01 00:13:00+00:00,True
3,2010-01-01 00:04:00+00:00,2010-01-01 00:14:00+00:00,True
4,2010-01-01 00:05:00+00:00,2010-01-01 00:15:00+00:00,True


In [4]:
# Cell 4 — Load the initial feature inputs.
#
# These values belong to completed bars and are known
# at decision_time. Nothing here uses future prices.
#
# We load selected columns to limit memory use.
# Quote-volume features will be considered separately:
# displayed quote size is not executed trading volume.

import numpy as np


BAR_INPUT_COLUMNS = [
    "mid_open",
    "mid_high",
    "mid_low",
    "mid_close",
    "spread_mean_units",
    "spread_max_units",
    "spread_close_units",
    "tick_count",
    "tick_imbalance",
]


# Check the saved schema before reading.
missing_columns = (
    set(BAR_INPUT_COLUMNS) - set(input_column_names)
)

assert not missing_columns, (
    f"Required input columns are missing: {sorted(missing_columns)}"
)


# Include timestamps temporarily to verify row alignment.
bar_inputs = pd.read_parquet(
    CANONICAL_INPUT_FILE,
    columns=["minute_start", *BAR_INPUT_COLUMNS],
)

assert len(bar_inputs) == len(canonical_timeline)

assert bar_inputs.index.equals(canonical_timeline.index)

assert bar_inputs["minute_start"].equals(
    canonical_timeline["minute_start"]
), "Price rows do not align with the timeline."


# The timeline already stores timestamps.
bar_inputs.drop(columns="minute_start", inplace=True)


# Check one column at a time instead of constructing
# a large temporary numeric matrix.
for column in BAR_INPUT_COLUMNS:
    assert np.isfinite(bar_inputs[column].to_numpy()).all(), (
        f"Non-finite values found in {column}."
    )

assert (bar_inputs["mid_close"] > 0).all()


input_memory_mib = (
    bar_inputs.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

print(f"Loaded rows: {len(bar_inputs):,}")
print(f"Numeric input columns: {len(BAR_INPUT_COLUMNS)}")
print(f"Input memory: {input_memory_mib:.2f} MiB")
print()
print("Feature-input alignment: PASS")

display(bar_inputs.head())

Loaded rows: 6,206,712
Numeric input columns: 9
Input memory: 426.18 MiB

Feature-input alignment: PASS


,mid_open,mid_high,mid_low,mid_close,spread_mean_units,spread_max_units,spread_close_units,tick_count,tick_imbalance
0,1.43288,1.43288,1.43272,1.43272,1.078571,1.3,1.0,14,-0.800000
1,1.43273,1.43290,1.43273,1.43290,1.046154,1.3,1.2,13,1.000000
2,1.43287,1.43287,1.43274,1.43287,1.200000,1.4,1.4,10,0.111111
3,1.43287,1.43287,1.43274,1.43279,1.080000,1.4,1.2,15,-0.166667
4,1.43288,1.43288,1.43282,1.43282,0.964706,1.2,1.0,17,-0.285714


In [5]:
# Cell 5 — Build backward-looking return features.
#
# Log returns express proportional price movement,
# avoiding dependence on an asset's pip size.
#
# Each return compares the current completed bar's close
# with an earlier completed bar's close.
# A return is unavailable if its lookback crosses a gap.
# Keep those values missing; do not replace them with zero.

RETURN_LOOKBACK_MINUTES = tuple(
    sorted({
        BAR_MINUTES,
        5,
        FORECAST_HORIZON_MINUTES,
    })
)

for minutes in RETURN_LOOKBACK_MINUTES:
    assert minutes >= BAR_MINUTES
    assert minutes % BAR_MINUTES == 0, (
        f"The {minutes}-minute lookback is incompatible "
        "with the configured bar duration."
    )


# Retain this Series for later volatility and target calculations.
log_mid_close = np.log(bar_inputs["mid_close"])

features = pd.DataFrame(index=canonical_timeline.index)

segment_lengths = (
    canonical_timeline.groupby("segment_id", sort=False).size()
)

return_feature_checks = []


for minutes in RETURN_LOOKBACK_MINUTES:
    lag_bars = minutes // BAR_MINUTES
    feature_name = f"log_return_{minutes}m"

    # Since segment IDs never repeat, matching IDs mean
    # the entire intervening lookback contains observed bars.
    valid_lookback = canonical_timeline["segment_id"].eq(
        canonical_timeline["segment_id"].shift(lag_bars)
    )

    features[feature_name] = (
        log_mid_close - log_mid_close.shift(lag_bars)
    ).where(valid_lookback)

    # Independently reconcile the number of valid returns.
    expected_count = int(
        (segment_lengths - lag_bars).clip(lower=0).sum()
    )

    observed_count = int(features[feature_name].notna().sum())

    assert observed_count == expected_count, (
        f"Lookback reconciliation failed for {feature_name}."
    )

    assert np.isfinite(
        features.loc[valid_lookback, feature_name].to_numpy()
    ).all(), f"Invalid numeric returns in {feature_name}."

    return_feature_checks.append({
        "feature": feature_name,
        "valid_rows": observed_count,
        "unavailable_rows": len(features) - observed_count,
    })


print("Backward-looking returns: PASS")
display(pd.DataFrame(return_feature_checks))
display(features.head())


# Missing values at segment starts are expected warmup.
# Target eligibility remains separate from feature eligibility.
del segment_lengths, valid_lookback

Backward-looking returns: PASS


,feature,valid_rows,unavailable_rows
0,log_return_1m,6191489,15223
1,log_return_5m,6140579,66133
2,log_return_10m,6087107,119605


,log_return_1m,log_return_5m,log_return_10m
0,NaN,NaN,NaN
1,0.000126,NaN,NaN
2,-0.000021,NaN,NaN
3,-0.000056,NaN,NaN
4,0.000021,NaN,NaN


In [6]:
# Cell 6 — Build backward-looking volatility features.
#
# Volatility is the standard deviation of completed-bar
# log returns within each lookback window.
#
# These values are not annualized and are not forecasts.
# A window requires the full number of returns:
# 30 one-minute returns require 31 consecutive closes.
#
# Zero volatility is valid. Missing history stays missing.

VOLATILITY_LOOKBACK_MINUTES = (30, 60)

base_return_column = f"log_return_{BAR_MINUTES}m"

segment_lengths = (
    canonical_timeline.groupby("segment_id", sort=False).size()
)

volatility_feature_checks = []


for minutes in VOLATILITY_LOOKBACK_MINUTES:
    assert minutes >= BAR_MINUTES
    assert minutes % BAR_MINUTES == 0, (
        f"The {minutes}-minute window is incompatible "
        "with the configured bar duration."
    )

    window_bars = minutes // BAR_MINUTES
    feature_name = f"log_return_std_{minutes}m"

    # Require the current close and the close preceding
    # the first return to belong to the same segment.
    valid_history = canonical_timeline["segment_id"].eq(
        canonical_timeline["segment_id"].shift(window_bars)
    )

    # Include the latest completed-bar return.
    # ddof=0 explicitly specifies population standard deviation.
    features[feature_name] = (
        features[base_return_column]
        .rolling(
            window=window_bars,
            min_periods=window_bars,
        )
        .std(ddof=0)
        .where(valid_history)
    )

    expected_count = int(
        (segment_lengths - window_bars).clip(lower=0).sum()
    )

    observed_count = int(features[feature_name].notna().sum())

    assert observed_count == expected_count, (
        f"Window reconciliation failed for {feature_name}."
    )

    valid_values = features.loc[valid_history, feature_name]

    assert np.isfinite(valid_values.to_numpy()).all()
    assert (valid_values >= 0).all()

    volatility_feature_checks.append({
        "feature": feature_name,
        "valid_rows": observed_count,
        "unavailable_rows": len(features) - observed_count,
    })


print("Backward-looking volatility: PASS")
display(pd.DataFrame(volatility_feature_checks))


# Show the first rows with both volatility windows available.
display(
    features.loc[
        features["log_return_std_60m"].notna()
    ].head()
)

del segment_lengths, valid_history, valid_values

Backward-looking volatility: PASS


,feature,valid_rows,unavailable_rows
0,log_return_std_30m,5918679,288033
1,log_return_std_60m,5728199,478513


,log_return_1m,log_return_5m,log_return_10m,log_return_std_30m,log_return_std_60m
60,-0.000154,-0.000202,0.000056,0.000141,0.000140
61,0.000049,-0.000154,0.000202,0.000139,0.000139
62,0.000105,-0.000049,0.000021,0.000124,0.000140
63,-0.000014,-0.000112,0.000007,0.000124,0.000140
64,-0.000293,-0.000307,-0.000307,0.000128,0.000145


In [7]:
# Cell 7 — Build completed-bar price-shape features.
#
# These features use only the latest completed bar.
# Log ratios avoid dependence on the asset's price level.
#
# Close location:
# 0 = closed at the low; 1 = closed at the high.
# A zero-range bar gets a neutral value of 0.5
# and a separate flag identifying that case.

bar_range = bar_inputs["mid_high"] - bar_inputs["mid_low"]
zero_range = bar_range.eq(0)

assert (bar_range >= 0).all()


features["bar_log_range"] = (
    np.log(bar_inputs["mid_high"])
    - np.log(bar_inputs["mid_low"])
)

features["bar_log_body"] = (
    log_mid_close - np.log(bar_inputs["mid_open"])
)


# Avoid division by zero; assign neutral location only
# to genuinely zero-range bars, not to missing history.
features["bar_close_location"] = (
    (bar_inputs["mid_close"] - bar_inputs["mid_low"])
    / bar_range.where(~zero_range)
).where(~zero_range, 0.5)

features["bar_zero_range"] = zero_range.astype("int8")


PRICE_SHAPE_COLUMNS = [
    "bar_log_range",
    "bar_log_body",
    "bar_close_location",
    "bar_zero_range",
]

for column in PRICE_SHAPE_COLUMNS:
    assert np.isfinite(features[column].to_numpy()).all(), (
        f"Invalid values in {column}."
    )

assert (features["bar_log_range"] >= 0).all()
assert features["bar_close_location"].between(0, 1).all()

assert (
    features["bar_log_body"].abs()
    <= features["bar_log_range"] + 1e-12
).all(), "Bar body exceeds its high-low range."


print("Completed-bar price shape: PASS")
print(f"Zero-range bars: {int(zero_range.sum()):,}")

display(features[PRICE_SHAPE_COLUMNS].head())

del bar_range, zero_range

Completed-bar price shape: PASS
Zero-range bars: 35,464


,bar_log_range,bar_log_body,bar_close_location,bar_zero_range
0,0.000112,-0.000112,0.000000,0
1,0.000119,0.000119,1.000000,0
2,0.000091,0.000000,1.000000,0
3,0.000091,-0.000056,0.384615,0
4,0.000042,-0.000042,0.000000,0


In [8]:
# Cell 8 — Build price-relative spread features.
#
# Convert reporting units back into price differences,
# then divide by the latest completed bar's midpoint close.
#
# Mean and maximum spreads use that same closing-price
# denominator; they are not averages of per-tick ratios.
#
# Do not cap wide spreads or remove locked quotes.
# These are market-state features, not execution-cost estimates.

REPORTING_UNIT_PRICE = float(
    instrument_config["reporting_unit"]
)

assert np.isfinite(REPORTING_UNIT_PRICE)
assert REPORTING_UNIT_PRICE > 0


SPREAD_FEATURE_SOURCES = {
    "spread_mean_relative": "spread_mean_units",
    "spread_max_relative": "spread_max_units",
    "spread_close_relative": "spread_close_units",
}


for feature_name, source_column in SPREAD_FEATURE_SOURCES.items():
    features[feature_name] = (
        bar_inputs[source_column]
        * REPORTING_UNIT_PRICE
        / bar_inputs["mid_close"]
    )

    assert np.isfinite(
        features[feature_name].to_numpy()
    ).all(), f"Invalid values in {feature_name}."

    assert (features[feature_name] >= 0).all()


# The maximum observed spread must cover both the mean
# and closing spread, allowing a small numeric tolerance.
for feature_name in [
    "spread_mean_relative",
    "spread_close_relative",
]:
    assert (
        features[feature_name]
        <= features["spread_max_relative"] + 1e-12
    ).all(), f"{feature_name} exceeds the maximum spread."


print("Price-relative spread features: PASS")

display(
    features[list(SPREAD_FEATURE_SOURCES)].head()
)

Price-relative spread features: PASS


,spread_mean_relative,spread_max_relative,spread_close_relative
0,0.000075,0.000091,0.000070
1,0.000073,0.000091,0.000084
2,0.000084,0.000098,0.000098
3,0.000075,0.000098,0.000084
4,0.000067,0.000084,0.000070


In [9]:
# Cell 9 — Build quote-activity features.
#
# Compare current quote activity with its preceding history.
# Exclude the current bar from that historical average.
#
# Tick imbalance measures upward versus downward midpoint
# updates. It does not identify aggressive buyers or sellers.
#
# The raw aggregation carried tick direction across gaps.
# Therefore, exclude imbalance on each segment's first bar.

ACTIVITY_LOOKBACK_MINUTES = min(
    VOLATILITY_LOOKBACK_MINUTES
)

assert ACTIVITY_LOOKBACK_MINUTES % BAR_MINUTES == 0

activity_window_bars = (
    ACTIVITY_LOOKBACK_MINUTES // BAR_MINUTES
)

activity_feature_name = (
    f"tick_activity_relative_{ACTIVITY_LOOKBACK_MINUTES}m"
)


# Log-transform quote updates per second.
# Absolute activity remains specific to the data feed.
features["log_tick_rate"] = np.log1p(
    bar_inputs["tick_count"] / BAR_DURATION.total_seconds()
)


# Average only the preceding completed bars.
previous_tick_count_mean = (
    bar_inputs["tick_count"]
    .rolling(
        window=activity_window_bars,
        min_periods=activity_window_bars,
    )
    .mean()
    .shift(1)
)

valid_activity_history = canonical_timeline["segment_id"].eq(
    canonical_timeline["segment_id"].shift(activity_window_bars)
)

features[activity_feature_name] = (
    bar_inputs["tick_count"] / previous_tick_count_mean
).where(valid_activity_history)


# Exclude the potentially gap-crossing direction comparison.
valid_imbalance_history = canonical_timeline["segment_id"].eq(
    canonical_timeline["segment_id"].shift(1)
)

features["tick_imbalance"] = (
    bar_inputs["tick_imbalance"]
    .where(valid_imbalance_history)
)


QUOTE_ACTIVITY_COLUMNS = [
    "log_tick_rate",
    activity_feature_name,
    "tick_imbalance",
]

for column in QUOTE_ACTIVITY_COLUMNS:
    assert np.isfinite(
        features[column].dropna().to_numpy()
    ).all(), f"Invalid values in {column}."

assert (features["log_tick_rate"] > 0).all()

assert features[activity_feature_name].notna().equals(
    valid_activity_history
)

assert (
    features.loc[valid_activity_history, activity_feature_name] > 0
).all()

assert features["tick_imbalance"].notna().equals(
    valid_imbalance_history
)

assert features["tick_imbalance"].dropna().between(-1, 1).all()


print("Quote-activity features: PASS")
print("Activity lookback:", ACTIVITY_LOOKBACK_MINUTES, "minutes")

display(
    features.loc[valid_activity_history, QUOTE_ACTIVITY_COLUMNS].head()
)

del previous_tick_count_mean
del valid_activity_history, valid_imbalance_history

Quote-activity features: PASS
Activity lookback: 30 minutes


,log_tick_rate,tick_activity_relative_30m,tick_imbalance
30,0.154151,0.795756,-0.333333
31,0.223144,1.206434,-0.230769
32,0.139762,0.720000,1.000000
33,0.236389,1.283422,0.000000
34,0.275103,1.520000,-0.444444


In [10]:
# Cell 10 — Freeze the feature specification and validate it.
#
# This combines:
# A. Reproducible feature settings.
# B. An independent reconstruction of the existing features.
# C. Future-data and gap-isolation tests.
# D. Full-dataset numeric and warmup checks.
#
# No training, fitting, target calculation, or file writing occurs here.

FEATURE_SETTINGS = {
    "bar_minutes": BAR_MINUTES,
    "reporting_unit_price": REPORTING_UNIT_PRICE,
    "return_lookback_minutes": list(RETURN_LOOKBACK_MINUTES),
    "volatility_lookback_minutes": list(VOLATILITY_LOOKBACK_MINUTES),
    "activity_lookback_minutes": ACTIVITY_LOOKBACK_MINUTES,
    "volatility_ddof": 0,
}


def rebuild_feature_frame(inputs, timeline, settings):
    # This function uses only supplied inputs and settings.
    # It does not access targets or future prices.
    bar_minutes = settings["bar_minutes"]
    segments = timeline["segment_id"]
    log_close = np.log(inputs["mid_close"])
    values = {}

    def history_available(lag):
        return segments.eq(segments.shift(lag))

    for minutes in settings["return_lookback_minutes"]:
        lag = minutes // bar_minutes
        values[f"log_return_{minutes}m"] = (
            log_close - log_close.shift(lag)
        ).where(history_available(lag))

    base_return = values[f"log_return_{bar_minutes}m"]

    for minutes in settings["volatility_lookback_minutes"]:
        window = minutes // bar_minutes
        values[f"log_return_std_{minutes}m"] = (
            base_return.rolling(window, min_periods=window)
            .std(ddof=settings["volatility_ddof"])
            .where(history_available(window))
        )

    price_range = inputs["mid_high"] - inputs["mid_low"]
    zero_range = price_range.eq(0)

    values["bar_log_range"] = (
        np.log(inputs["mid_high"]) - np.log(inputs["mid_low"])
    )
    values["bar_log_body"] = (
        log_close - np.log(inputs["mid_open"])
    )
    values["bar_close_location"] = (
        (inputs["mid_close"] - inputs["mid_low"])
        / price_range.where(~zero_range)
    ).where(~zero_range, 0.5)
    values["bar_zero_range"] = zero_range.astype("int8")

    for kind in ["mean", "max", "close"]:
        values[f"spread_{kind}_relative"] = (
            inputs[f"spread_{kind}_units"]
            * settings["reporting_unit_price"]
            / inputs["mid_close"]
        )

    values["log_tick_rate"] = np.log1p(
        inputs["tick_count"] / (bar_minutes * 60)
    )

    minutes = settings["activity_lookback_minutes"]
    window = minutes // bar_minutes
    previous_activity = (
        inputs["tick_count"]
        .rolling(window, min_periods=window)
        .mean()
        .shift(1)
    )

    values[f"tick_activity_relative_{minutes}m"] = (
        inputs["tick_count"] / previous_activity
    ).where(history_available(window))

    values["tick_imbalance"] = (
        inputs["tick_imbalance"].where(history_available(1))
    )

    return pd.DataFrame(values, index=inputs.index)


# A — Check the reconstruction against already-built features.
# Only a manageable prefix is reconstructed in memory.
prefix_rows = min(20_000, len(features))

reconstructed_prefix = rebuild_feature_frame(
    bar_inputs.iloc[:prefix_rows],
    canonical_timeline.iloc[:prefix_rows],
    FEATURE_SETTINGS,
)

pd.testing.assert_frame_equal(
    features.iloc[:prefix_rows],
    reconstructed_prefix,
    check_exact=False,
    rtol=1e-9,
    atol=1e-12,
)

print("Feature reconstruction: PASS")


# B — Create synthetic data with two continuous segments.
# This tests behaviour independently of this particular CSV.
maximum_history_minutes = max(
    *FEATURE_SETTINGS["return_lookback_minutes"],
    *FEATURE_SETTINGS["volatility_lookback_minutes"],
    FEATURE_SETTINGS["activity_lookback_minutes"],
    BAR_MINUTES,
)
MAXIMUM_HISTORY_BARS = maximum_history_minutes // BAR_MINUTES

test_segment_length = MAXIMUM_HISTORY_BARS + 30
test_rows = test_segment_length * 2
test_prices = 1.0 + np.arange(test_rows) * 0.0001

test_inputs = pd.DataFrame({
    "mid_open": test_prices - 0.00005,
    "mid_high": test_prices + 0.0001,
    "mid_low": test_prices - 0.0001,
    "mid_close": test_prices,
    "spread_mean_units": np.full(test_rows, 1.0),
    "spread_max_units": np.full(test_rows, 1.5),
    "spread_close_units": np.full(test_rows, 1.0),
    "tick_count": 10 + np.arange(test_rows) % 17,
    "tick_imbalance": np.zeros(test_rows),
})

test_minutes = pd.Series(
    pd.date_range(
        "2020-01-01",
        periods=test_rows,
        freq=BAR_DURATION,
        tz="UTC",
    )
)

# Insert an actual timestamp gap before the second segment.
test_minutes.iloc[test_segment_length:] = (
    test_minutes.iloc[test_segment_length:] + 30 * BAR_DURATION
)

test_timeline = pd.DataFrame({
    "minute_start": test_minutes,
    "decision_time": test_minutes + BAR_DURATION,
})
test_timeline["segment_id"] = (
    test_minutes.diff().ne(BAR_DURATION).cumsum().astype("int32") - 1
)

test_features = rebuild_feature_frame(
    test_inputs, test_timeline, FEATURE_SETTINGS
)


# C — Change future observations drastically.
# Features before the change must remain identical.
change_start = MAXIMUM_HISTORY_BARS + 5
changed_inputs = test_inputs.copy()

price_columns = ["mid_open", "mid_high", "mid_low", "mid_close"]
changed_inputs.loc[change_start:, price_columns] = (
    changed_inputs.loc[change_start:, price_columns] * 1.5
)
changed_inputs.loc[change_start:, "tick_count"] *= 3
changed_inputs.loc[change_start:, "tick_imbalance"] = 0.75

changed_features = rebuild_feature_frame(
    changed_inputs, test_timeline, FEATURE_SETTINGS
)

pd.testing.assert_frame_equal(
    test_features.iloc[:change_start],
    changed_features.iloc[:change_start],
    check_exact=False,
    rtol=1e-9,
    atol=1e-12,
)

print("Future-data causality test: PASS")


# D — Recalculate the second segment completely independently.
# Data before the gap must not influence its feature values.
fresh_segment_features = rebuild_feature_frame(
    test_inputs.iloc[test_segment_length:].reset_index(drop=True),
    test_timeline.iloc[test_segment_length:].reset_index(drop=True),
    FEATURE_SETTINGS,
)

pd.testing.assert_frame_equal(
    test_features.iloc[test_segment_length:].reset_index(drop=True),
    fresh_segment_features,
    check_exact=False,
    rtol=1e-9,
    atol=1e-12,
)

print("Gap-isolation test: PASS")


# E — Validate availability and numeric values across ALL rows.
FEATURE_COLUMNS = list(features.columns)

history_requirements = {column: 0 for column in FEATURE_COLUMNS}

for minutes in RETURN_LOOKBACK_MINUTES:
    history_requirements[f"log_return_{minutes}m"] = (
        minutes // BAR_MINUTES
    )

for minutes in VOLATILITY_LOOKBACK_MINUTES:
    history_requirements[f"log_return_std_{minutes}m"] = (
        minutes // BAR_MINUTES
    )

history_requirements[activity_feature_name] = activity_window_bars
history_requirements["tick_imbalance"] = 1

for column, required_history in history_requirements.items():
    if required_history == 0:
        expected_available = pd.Series(
            True, index=canonical_timeline.index
        )
    else:
        expected_available = canonical_timeline["segment_id"].eq(
            canonical_timeline["segment_id"].shift(required_history)
        )

    assert features[column].notna().equals(expected_available), (
        f"Unexpected missing-value pattern in {column}."
    )
    assert np.isfinite(
        features[column].dropna().to_numpy()
    ).all(), f"Non-finite values in {column}."


features_ready = features.notna().all(axis=1)

expected_ready = canonical_timeline["segment_id"].eq(
    canonical_timeline["segment_id"].shift(MAXIMUM_HISTORY_BARS)
)

assert features_ready.equals(expected_ready)

print("Full feature validation: PASS")
print(f"Model-input features: {len(FEATURE_COLUMNS)}")
print(f"Required preceding history: {maximum_history_minutes} minutes")
print(f"Feature-ready rows: {int(features_ready.sum()):,}")
print(
    "Feature memory:",
    f"{features.memory_usage(deep=True).sum() / 1024**2:.2f} MiB",
)


# Release test objects. Retain settings, features, and readiness.
del reconstructed_prefix, test_inputs, test_timeline, test_minutes
del test_features, changed_inputs, changed_features
del fresh_segment_features, test_prices
del expected_available, expected_ready

Feature reconstruction: PASS
Future-data causality test: PASS
Gap-isolation test: PASS
Full feature validation: PASS
Model-input features: 15
Required preceding history: 60 minutes
Feature-ready rows: 5,728,199
Feature memory: 668.87 MiB


In [11]:
# Cell 11 — Build targets separately from model inputs.
#
# Primary forecast: the configured 10-minute horizon.
# Auxiliary target: gross midpoint movement over 5 minutes.
#
# The auxiliary target is NOT executable P&L:
# it does not include bid/ask execution, quote size, delay,
# commissions, or the agreed execution slippage.
#
# Targets use completed-bar closes at the specified endpoints,
# not guaranteed executable quotes exactly at those timestamps.
#
# Initial neutral band:
# max(volatility component, current-spread component).
#
# Volatility uses a square-root-of-time scaling approximation.
# This is a label-definition convention, not a calibrated forecast.
#
# Multipliers must later be assessed using training/validation
# data only, then frozen before final holdout evaluation.

TARGET_SETTINGS = {
    "primary_horizon_minutes": FORECAST_HORIZON_MINUTES,
    "auxiliary_horizon_minutes": 5,
    "volatility_window_minutes": max(VOLATILITY_LOOKBACK_MINUTES),
    "volatility_multiplier": 1.0,
    "spread_multiplier": 1.0,
    "band_units": "simple_return_basis_points",
    "direction_encoding": {"DOWN": -1, "FLAT": 0, "UP": 1},
}

PRIMARY_HORIZON = TARGET_SETTINGS["primary_horizon_minutes"]
AUXILIARY_HORIZON = TARGET_SETTINGS["auxiliary_horizon_minutes"]

assert 0 < AUXILIARY_HORIZON <= PRIMARY_HORIZON

for minutes in [PRIMARY_HORIZON, AUXILIARY_HORIZON]:
    assert minutes % BAR_MINUTES == 0

for setting in ["volatility_multiplier", "spread_multiplier"]:
    assert np.isfinite(TARGET_SETTINGS[setting])
    assert TARGET_SETTINGS[setting] > 0


targets = pd.DataFrame(index=canonical_timeline.index)
window_valid_by_horizon = {}

for minutes in sorted({PRIMARY_HORIZON, AUXILIARY_HORIZON}):
    horizon_bars = minutes // BAR_MINUTES
    expected_endpoint = (
        canonical_timeline["decision_time"]
        + pd.Timedelta(minutes=minutes)
    )

    valid_window = (
        canonical_timeline["segment_id"].eq(
            canonical_timeline["segment_id"].shift(-horizon_bars)
        )
        & canonical_timeline["decision_time"]
        .shift(-horizon_bars)
        .eq(expected_endpoint)
    )

    future_log_return = (
        log_mid_close.shift(-horizon_bars) - log_mid_close
    ).where(valid_window)

    targets[f"target_log_return_{minutes}m"] = future_log_return

    # expm1 converts log returns into exact proportional returns.
    # Multiplying by 10,000 expresses those returns in basis points.
    targets[f"target_return_bps_{minutes}m"] = (
        np.expm1(future_log_return) * 10_000
    )

    window_valid_by_horizon[minutes] = valid_window

    assert future_log_return.notna().equals(valid_window)


assert window_valid_by_horizon[PRIMARY_HORIZON].equals(
    canonical_timeline["target_window_valid"]
)


# Compute the neutral band using information known at decision time.
volatility_column = (
    f"log_return_std_{TARGET_SETTINGS['volatility_window_minutes']}m"
)

horizon_log_sigma = (
    features[volatility_column]
    * np.sqrt(PRIMARY_HORIZON // BAR_MINUTES)
)

volatility_band_bps = (
    np.expm1(
        TARGET_SETTINGS["volatility_multiplier"] * horizon_log_sigma
    )
    * 10_000
)

spread_band_bps = (
    features["spread_close_relative"]
    * TARGET_SETTINGS["spread_multiplier"]
    * 10_000
)

# The band is symmetric in simple-return basis points.
# Missing volatility history produces a missing band.
targets["neutral_band_bps"] = np.maximum(
    volatility_band_bps, spread_band_bps
)

assert (
    targets["neutral_band_bps"].dropna() >= 0
).all()


eligible_rows = (
    features_ready
    & window_valid_by_horizon[PRIMARY_HORIZON]
    & window_valid_by_horizon[AUXILIARY_HORIZON]
    & targets["neutral_band_bps"].notna()
)

PRIMARY_RETURN_COLUMN = f"target_return_bps_{PRIMARY_HORIZON}m"
DIRECTION_COLUMN = f"target_direction_{PRIMARY_HORIZON}m"

# Missing labels stay missing, rather than being labelled FLAT.
targets[DIRECTION_COLUMN] = pd.Series(
    pd.NA, index=targets.index, dtype="Int8"
)

targets.loc[eligible_rows, DIRECTION_COLUMN] = 0

targets.loc[
    eligible_rows
    & targets[PRIMARY_RETURN_COLUMN].gt(targets["neutral_band_bps"]),
    DIRECTION_COLUMN,
] = 1

targets.loc[
    eligible_rows
    & targets[PRIMARY_RETURN_COLUMN].lt(-targets["neutral_band_bps"]),
    DIRECTION_COLUMN,
] = -1


# Independently reconcile eligibility using segment lengths.
# A segment loses its initial feature warmup and final target window.
segment_lengths = (
    canonical_timeline.groupby("segment_id", sort=False).size()
)

expected_eligible_rows = int(
    (
        segment_lengths
        - MAXIMUM_HISTORY_BARS
        - PRIMARY_HORIZON // BAR_MINUTES
    )
    .clip(lower=0)
    .sum()
)

eligible_row_count = int(eligible_rows.sum())

assert eligible_row_count == expected_eligible_rows
assert eligible_row_count > 0

assert targets[DIRECTION_COLUMN].notna().equals(eligible_rows)
assert targets.loc[
    eligible_rows, DIRECTION_COLUMN
].isin([-1, 0, 1]).all()

TARGET_COLUMNS = list(targets.columns)

for column in TARGET_COLUMNS:
    assert np.isfinite(
        targets.loc[eligible_rows, column].to_numpy(dtype="float64")
    ).all(), f"Invalid eligible target values in {column}."


# Check return calculations directly against endpoint prices.
# This is a structural test, not a performance evaluation.
eligible_positions = np.flatnonzero(eligible_rows.to_numpy())
check_positions = eligible_positions[
    np.linspace(
        0, len(eligible_positions) - 1, num=64, dtype=int
    )
]

for minutes in sorted({PRIMARY_HORIZON, AUXILIARY_HORIZON}):
    horizon_bars = minutes // BAR_MINUTES

    expected_bps = (
        bar_inputs["mid_close"].iloc[
            check_positions + horizon_bars
        ].to_numpy()
        / bar_inputs["mid_close"].iloc[check_positions].to_numpy()
        - 1
    ) * 10_000

    np.testing.assert_allclose(
        targets[f"target_return_bps_{minutes}m"]
        .iloc[check_positions].to_numpy(),
        expected_bps,
        rtol=1e-10,
        atol=1e-8,
    )


canonical_timeline["auxiliary_target_time"] = (
    canonical_timeline["decision_time"]
    + pd.Timedelta(minutes=AUXILIARY_HORIZON)
)

print("Endpoint return checks: PASS")
print("Scaled-band target validation: PASS")
print("Eligibility reconciliation: PASS")
print(f"Primary forecast horizon: {PRIMARY_HORIZON} minutes")
print(f"Auxiliary return horizon: {AUXILIARY_HORIZON} minutes")
print(
    "Band multipliers:",
    f"volatility={TARGET_SETTINGS['volatility_multiplier']},",
    f"spread={TARGET_SETTINGS['spread_multiplier']}",
)
print(f"Eligible rows: {eligible_row_count:,}")
print(f"Excluded rows: {len(features) - eligible_row_count:,}")


# Preview only the earliest eligible observations.
preview_positions = eligible_positions[:5]

display(
    pd.concat(
        [
            canonical_timeline.iloc[preview_positions][
                ["decision_time", "target_time"]
            ],
            targets.iloc[preview_positions][
                [
                    PRIMARY_RETURN_COLUMN,
                    "neutral_band_bps",
                    DIRECTION_COLUMN,
                ]
            ],
        ],
        axis=1,
    )
)

# FLAT describes the future movement relative to the band.
# It is not itself a NO TRADE instruction.
# Trading probability and expected-net-profit gates come later.
del segment_lengths, eligible_positions, check_positions
del horizon_log_sigma, volatility_band_bps, spread_band_bps

Endpoint return checks: PASS
Scaled-band target validation: PASS
Eligibility reconciliation: PASS
Primary forecast horizon: 10 minutes
Auxiliary return horizon: 5 minutes
Band multipliers: volatility=1.0, spread=1.0
Eligible rows: 5,672,911
Excluded rows: 533,801


,decision_time,target_time,target_return_bps_10m,neutral_band_bps,target_direction_10m
60,2010-01-01 01:01:00+00:00,2010-01-01 01:11:00+00:00,0.418816,4.428660,0
61,2010-01-01 01:02:00+00:00,2010-01-01 01:12:00+00:00,-0.418796,4.402808,0
62,2010-01-01 01:03:00+00:00,2010-01-01 01:13:00+00:00,0.418752,4.423409,0
63,2010-01-01 01:04:00+00:00,2010-01-01 01:14:00+00:00,-1.884409,4.417914,0
64,2010-01-01 01:05:00+00:00,2010-01-01 01:15:00+00:00,3.630296,4.573745,0


In [ ]:
# Cell 12 — Export the eligible dataset and its manifest.
#
# This combines:
# A. Safe output preparation.
# B. Bounded-memory Parquet writing.
# C. Full read-back verification.
# D. Configuration and provenance export.
#
# Only explicitly listed FEATURE_COLUMNS are model inputs.
# Target and metadata columns must never enter the input matrix.
#
# Existing outputs are never overwritten automatically.
# On failure, partial files remain available for inspection.
# Files are promoted individually only after validation succeeds.

import hashlib
import json
import re
import pyarrow as pa
import pyarrow.parquet as pq


def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(8 * 1024 * 1024)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


def row_hash_bytes(frame):
    # Hash every row, excluding the temporary pandas index.
    # Streaming these bytes allows different batch boundaries
    # during writing and reading without changing the checksum.
    return pd.util.hash_pandas_object(
        frame, index=False
    ).to_numpy(dtype="uint64").tobytes()


METADATA_COLUMNS = [
    "minute_start",
    "decision_time",
    "target_time",
    "auxiliary_target_time",
    "segment_id",
]

OUTPUT_COLUMNS = [
    *METADATA_COLUMNS,
    *FEATURE_COLUMNS,
    *TARGET_COLUMNS,
]

assert len(OUTPUT_COLUMNS) == len(set(OUTPUT_COLUMNS))

symbol_slug = re.sub(
    r"[^a-z0-9_-]+",
    "_",
    str(instrument_config["symbol"]).lower(),
).strip("_")

assert symbol_slug

first_date = pd.Timestamp(
    build_summary["dataset"]["first_bar_utc"]
).date().isoformat()

last_date = pd.Timestamp(
    build_summary["dataset"]["last_bar_utc"]
).date().isoformat()

output_stem = (
    f"{symbol_slug}_features_targets_{PRIMARY_HORIZON}m"
    f"_{first_date}_{last_date}"
)

FEATURE_TARGET_OUTPUT_FILE = (
    PROCESSED_DATA_DIR / f"{output_stem}.parquet"
)
TEMP_FEATURE_TARGET_FILE = (
    PROCESSED_DATA_DIR / f"{output_stem}.partial.parquet"
)
FEATURE_TARGET_MANIFEST_FILE = (
    REPORT_TABLES_DIR / f"{output_stem}_manifest.json"
)
TEMP_FEATURE_TARGET_MANIFEST_FILE = (
    REPORT_TABLES_DIR / f"{output_stem}_manifest.partial.json"
)

for path in [
    FEATURE_TARGET_OUTPUT_FILE,
    TEMP_FEATURE_TARGET_FILE,
    FEATURE_TARGET_MANIFEST_FILE,
    TEMP_FEATURE_TARGET_MANIFEST_FILE,
]:
    if path.exists():
        raise FileExistsError(
            f"Output already exists; inspect before rerunning: {path}"
        )

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

source_stat = CANONICAL_INPUT_FILE.stat()
source_snapshot = (source_stat.st_size, source_stat.st_mtime_ns)

assert source_stat.st_size == build_summary["dataset"]["size_bytes"]

EXPORT_BATCH_ROWS = 100_000
written_rows = 0
output_schema = None
expected_digest = hashlib.sha256()


# Write eligible rows in manageable batches.
# Do not assemble another full-sized combined DataFrame.
with TEMP_FEATURE_TARGET_FILE.open("xb") as output_handle:
    writer = None

    try:
        for start in range(0, len(features), EXPORT_BATCH_ROWS):
            stop = min(start + EXPORT_BATCH_ROWS, len(features))
            batch_mask = eligible_rows.iloc[start:stop]

            if not batch_mask.any():
                continue

            output_batch = pd.concat(
                [
                    canonical_timeline.iloc[start:stop][
                        METADATA_COLUMNS
                    ].loc[batch_mask],
                    features.iloc[start:stop][
                        FEATURE_COLUMNS
                    ].loc[batch_mask],
                    targets.iloc[start:stop][
                        TARGET_COLUMNS
                    ].loc[batch_mask],
                ],
                axis=1,
            ).reset_index(drop=True)

            assert list(output_batch.columns) == OUTPUT_COLUMNS
            assert not output_batch.isna().any().any()

            if output_schema is None:
                arrow_table = pa.Table.from_pandas(
                    output_batch, preserve_index=False
                )
                output_schema = arrow_table.schema

                writer = pq.ParquetWriter(
                    output_handle,
                    output_schema,
                    compression="zstd",
                )
            else:
                arrow_table = pa.Table.from_pandas(
                    output_batch,
                    schema=output_schema,
                    preserve_index=False,
                    safe=True,
                )

            writer.write_table(arrow_table)
            expected_digest.update(row_hash_bytes(output_batch))
            written_rows += len(output_batch)

    finally:
        if writer is not None:
            writer.close()


assert written_rows == eligible_row_count
print(f"Temporary export written: {written_rows:,} rows")


# Independently read every saved row.
saved_rows = 0
saved_digest = hashlib.sha256()
previous_decision_time = None
saved_first_decision = None
saved_last_decision = None

with pq.ParquetFile(TEMP_FEATURE_TARGET_FILE) as saved_file:
    assert saved_file.metadata.num_rows == eligible_row_count
    assert saved_file.schema_arrow.names == OUTPUT_COLUMNS
    assert saved_file.schema_arrow.equals(
        output_schema, check_metadata=True
    )

    for saved_batch in saved_file.iter_batches(
        batch_size=EXPORT_BATCH_ROWS
    ):
        saved = saved_batch.to_pandas()

        assert not saved.isna().any().any()

        numeric_values = saved.select_dtypes(
            include="number"
        ).to_numpy(dtype="float64")

        assert np.isfinite(numeric_values).all()

        decision_times = saved["decision_time"]

        assert decision_times.is_monotonic_increasing
        assert decision_times.is_unique

        if previous_decision_time is not None:
            assert decision_times.iloc[0] > previous_decision_time

        assert (
            decision_times == saved["minute_start"] + BAR_DURATION
        ).all()

        assert (
            saved["target_time"]
            == decision_times + pd.Timedelta(minutes=PRIMARY_HORIZON)
        ).all()

        assert (
            saved["auxiliary_target_time"]
            == decision_times + pd.Timedelta(minutes=AUXILIARY_HORIZON)
        ).all()

        assert (saved["neutral_band_bps"] >= 0).all()

        expected_directions = np.select(
            [
                saved[PRIMARY_RETURN_COLUMN]
                > saved["neutral_band_bps"],
                saved[PRIMARY_RETURN_COLUMN]
                < -saved["neutral_band_bps"],
            ],
            [1, -1],
            default=0,
        )

        np.testing.assert_array_equal(
            saved[DIRECTION_COLUMN].to_numpy(dtype="int8"),
            expected_directions,
        )

        if saved_first_decision is None:
            saved_first_decision = decision_times.iloc[0]

        saved_last_decision = decision_times.iloc[-1]
        previous_decision_time = saved_last_decision

        saved_digest.update(row_hash_bytes(saved))
        saved_rows += len(saved)

        if saved_rows % 2_000_000 == 0:
            print(f"Read-back validated: {saved_rows:,} rows")


assert saved_rows == eligible_row_count
assert saved_digest.hexdigest() == expected_digest.hexdigest(), (
    "Saved row values or ordering changed during serialization."
)

print("Full read-back validation: PASS")
print("Row-value and ordering checksum: PASS")


# Record file identities for reproducibility.
source_sha256 = file_sha256(CANONICAL_INPUT_FILE)
output_sha256 = file_sha256(TEMP_FEATURE_TARGET_FILE)

source_stat_after = CANONICAL_INPUT_FILE.stat()

assert (
    source_stat_after.st_size,
    source_stat_after.st_mtime_ns,
) == source_snapshot, "Canonical input changed during export."


manifest = {
    "status": build_summary["status"],
    "pipeline_version": "03_features_and_targets_v1",
    "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "instrument_config": instrument_config,
    "feature_settings": FEATURE_SETTINGS,
    "target_settings": TARGET_SETTINGS,
    "column_roles": {
        "model_inputs": FEATURE_COLUMNS,
        "targets_and_label_audit": TARGET_COLUMNS,
        "metadata": METADATA_COLUMNS,
    },
    "source": {
        "relative_path": str(
            CANONICAL_INPUT_FILE.relative_to(PROJECT_ROOT)
        ),
        "sha256": source_sha256,
        "build_summary_relative_path": str(
            BUILD_SUMMARY_FILE.relative_to(PROJECT_ROOT)
        ),
        "build_summary_sha256": file_sha256(BUILD_SUMMARY_FILE),
        "observed_bars": input_row_count,
        "recorded_gap_count": build_summary["coverage"]["gap_count"],
        "isolated_missing_clock_hours": (
            build_summary["coverage"]["isolated_missing_clock_hours"]
        ),
    },
    "output": {
        "relative_path": str(
            FEATURE_TARGET_OUTPUT_FILE.relative_to(PROJECT_ROOT)
        ),
        "sha256": output_sha256,
        "row_value_checksum": saved_digest.hexdigest(),
        "rows": saved_rows,
        "columns": len(OUTPUT_COLUMNS),
        "size_bytes": TEMP_FEATURE_TARGET_FILE.stat().st_size,
        "first_decision_time_utc": saved_first_decision.isoformat(),
        "last_decision_time_utc": saved_last_decision.isoformat(),
        "excluded_source_rows": input_row_count - saved_rows,
        "required_preceding_history_bars": MAXIMUM_HISTORY_BARS,
        "schema": {
            field.name: str(field.type) for field in output_schema
        },
    },
    "neutral_band_formula": (
        "max(10000 * expm1(volatility_multiplier * "
        "historical_log_return_std * sqrt(primary_horizon_bars)), "
        "10000 * spread_multiplier * closing_spread_relative)"
    ),
    "validation": {
        "feature_reconstruction": True,
        "future_data_causality_test": True,
        "gap_isolation_test": True,
        "full_feature_validation": True,
        "target_endpoint_checks": True,
        "eligibility_reconciliation": True,
        "full_export_readback": True,
        "row_value_checksum_match": True,
    },
    "limitations_and_next_stage_rules": [
        "Coverage warnings remain; missing source hours were not repaired.",
        "Band multipliers are initial settings, not optimized parameters.",
        "Square-root-of-time volatility scaling is an approximation.",
        "Targets are gross midpoint returns, not executable trade P&L.",
        "FLAT is a movement label, not an automatic NO TRADE decision.",
        "Only model_inputs may enter the model feature matrix.",
        "Fit preprocessing and models using training data only.",
        "Use chronological splits and purge labels overlapping boundaries.",
        "Select settings on training/validation data; freeze before holdout.",
        "Execution simulation and probability calibration are not implemented.",
    ],
}


# Stage and verify the manifest before promoting either file.
with TEMP_FEATURE_TARGET_MANIFEST_FILE.open(
    "x", encoding="utf-8"
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

with TEMP_FEATURE_TARGET_MANIFEST_FILE.open(
    "r", encoding="utf-8"
) as handle:
    assert json.load(handle) == manifest


# On Windows, rename refuses to replace an existing destination.
# Recheck explicitly before promoting the validated outputs.
assert not FEATURE_TARGET_OUTPUT_FILE.exists()
assert not FEATURE_TARGET_MANIFEST_FILE.exists()

TEMP_FEATURE_TARGET_FILE.rename(FEATURE_TARGET_OUTPUT_FILE)
TEMP_FEATURE_TARGET_MANIFEST_FILE.rename(
    FEATURE_TARGET_MANIFEST_FILE
)

assert FEATURE_TARGET_OUTPUT_FILE.exists()
assert FEATURE_TARGET_MANIFEST_FILE.exists()
assert not TEMP_FEATURE_TARGET_FILE.exists()
assert not TEMP_FEATURE_TARGET_MANIFEST_FILE.exists()


print()
print("Features and targets build: COMPLETE")
print("Status:", manifest["status"])
print(f"Saved rows: {saved_rows:,}")
print(f"Model-input features: {len(FEATURE_COLUMNS)}")
print(f"Total saved columns: {len(OUTPUT_COLUMNS)}")
print(
    "Output size:",
    f"{FEATURE_TARGET_OUTPUT_FILE.stat().st_size / 1024**2:.2f} MiB",
)
print("Dataset:", FEATURE_TARGET_OUTPUT_FILE)
print("Manifest:", FEATURE_TARGET_MANIFEST_FILE)


# Release export buffers, retaining notebook results and settings.
del output_batch, arrow_table, saved, numeric_values

Temporary export written: 5,672,911 rows
